# RapidMatch demo

Build a control group whose pre-campaign features look like the campaign target group.

This notebook:
1. Builds a small synthetic dataset with a known imbalance
2. Runs `ControlMatcher`
3. Inspects coverage, pair quality, and a simple before/after mean check

Run from the repo root so `import rapidmatch` resolves.

In [1]:
from pathlib import Path
import sys

root = Path.cwd()
if not (root / "rapidmatch").exists() and (root.parent / "rapidmatch").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

import numpy as np
import pandas as pd

from rapidmatch import ControlMatcher, MatchConfig

## 1. Synthetic data

200 targets clustered around income 70k / age 42.

800 controls: half from the same distribution (easy matches), half younger and poorer (deliberate mismatch). A naive random control sample would be pulled toward the poor/young mass. Matching should prefer the lookalike half.

In [2]:
rng = np.random.default_rng(42)
n_target, n_like, n_other = 200, 400, 400

target = pd.DataFrame({
    "id": np.arange(n_target, dtype=int),
    "income": rng.normal(70000, 8000, n_target),
    "age": rng.normal(42, 6, n_target),
    "region": rng.choice(["N", "S", "E", "W"], n_target),
    "is_target": 1,
})
like = pd.DataFrame({
    "id": np.arange(n_target, n_target + n_like, dtype=int),
    "income": rng.normal(70000, 8000, n_like),
    "age": rng.normal(42, 6, n_like),
    "region": rng.choice(["N", "S", "E", "W"], n_like),
    "is_target": 0,
})
other = pd.DataFrame({
    "id": np.arange(n_target + n_like, n_target + n_like + n_other, dtype=int),
    "income": rng.normal(45000, 9000, n_other),
    "age": rng.normal(30, 7, n_other),
    "region": rng.choice(["N", "S", "E", "W"], n_other),
    "is_target": 0,
})
df = pd.concat([target, like, other], ignore_index=True)
df.head()

,id,income,age,region,is_target
0,0,72437.736638,44.025447,S,1
1,1,61680.127150,50.444891,N,1
2,2,76003.609566,42.543509,W,1
3,3,77524.517731,45.863633,W,1
4,4,54391.718491,29.698967,W,1


In [3]:
df.groupby("is_target")[["income", "age"]].mean().round(1)

,income,age
is_target,,
0,56518.7,35.8
1,69756.4,42.1


## 2. Configure and run

- `match_vars` are used both to stratify and to score distance
- `weights` boost income over age in the distance
- `tolerance=0.2` keeps pairs at or above the 20th percentile of accepted strengths (fairly loose, good for a first look)
- `min_control_pool_size=3` flags thin strata but still matches them

In [4]:
config = MatchConfig(
    match_vars=["income", "age", "region"],
    treatment_col="is_target",
    id_col="id",
    weights={"income": 1.5, "age": 1.0},
    n=1,
    tolerance=0.2,
    min_control_pool_size=3,
    n_bins=4,
)
matcher = ControlMatcher(config)
result = matcher.fit_match(df)
result.coverage_summary

{'n_target': 200,
 'n_matched': 149,
 'n_no_control': 0,
 'n_below_tolerance': 37,
 'n_unmatched': 14,
 'n_thin_stratum': 13,
 'pct_matched': 0.745,
 'pct_no_control': 0.0,
 'pct_below_tolerance': 0.185,
 'tolerance_cutoff': 0.5656685620472034}

## 3. Who got a match?

`match_status`:
- `matched` — kept after tolerance
- `below_tolerance` — had a candidate, too weak
- `no_control_available` — empty control cell
- `unmatched` — eligible, lost the control to a stronger pair

`thin_stratum` is a separate flag and can sit on a `matched` row.

In [5]:
from collections import Counter

# `targets` and `pairs` are pyarrow.Table; learn the Arrow API or
# call `.to_pandas()` at the display boundary when convenient.
Counter(result.targets["match_status"].to_pylist())

Counter({'matched': 149, 'below_tolerance': 37, 'unmatched': 14})

In [6]:
import pyarrow.compute as pc

# Arrow filter, then hand off to pandas for the analysis cells below.
matched = result.pairs.filter(pc.equal(result.pairs["match_status"], "matched")).to_pandas()
matched.sort_values("match_strength", ascending=False).head(10)

,target_id,control_id,target_rm_id,control_rm_id,stratum,match_strength,match_rank,match_status,thin_stratum
21,24.0,389.0,25,390,1|2|N|false|false,0.977265,1,matched,False
58,75.0,253.0,76,254,3|2|N|false|false,0.972336,1,matched,False
74,98.0,255.0,99,256,2|0|E|false|false,0.959597,1,matched,False
1,2.0,441.0,3,442,3|2|W|false|false,0.946910,1,matched,False
93,125.0,561.0,126,562,2|3|E|false|false,0.946314,1,matched,False
147,198.0,530.0,199,531,2|2|S|false|false,0.943012,1,matched,False
88,116.0,303.0,117,304,2|2|W|false|false,0.937087,1,matched,False
73,97.0,225.0,98,226,0|1|E|false|false,0.932028,1,matched,False
23,28.0,435.0,29,436,2|0|W|false|false,0.930424,1,matched,False
117,161.0,539.0,162,540,0|2|S|false|false,0.929790,1,matched,False


## 4. Did matching actually balance income and age?

Compare target means against (a) a random control sample of the same size and (b) the matched controls.

In [7]:
target_rows = df[df["is_target"] == 1]
control_rows = df[df["is_target"] == 0]
random_ctrl = control_rows.sample(n=len(matched), random_state=0)
matched_ctrl = df.set_index("id").loc[matched["control_id"].astype(int)]

def means(frame, label):
    return pd.Series({
        "income": frame["income"].mean(),
        "age": frame["age"].mean(),
        "n": len(frame),
    }, name=label)

balance = pd.concat([
    means(target_rows, "target"),
    means(random_ctrl, "random_control"),
    means(matched_ctrl, "matched_control"),
], axis=1).T
balance.round(2)

,income,age,n
target,69756.39,42.12,200.0
random_control,54559.45,35.17,149.0
matched_control,69336.68,41.88,149.0


Matched control means should sit much closer to the target than the random sample. Strength should all lie in (0, 1].

In [8]:
print("n matched pairs:", len(matched))
print("unique controls:", matched["control_id"].nunique())
print("strength min/max:", matched["match_strength"].min(), matched["match_strength"].max())
print("thin among matched:", int(matched["thin_stratum"].sum()))
print("bin edges used:", matcher.bin_edges)
print("tolerance cutoff:", result.cutoff)

n matched pairs: 149
unique controls: 149
strength min/max: 0.5656685620472034 0.9772653900757288
thin among matched: 6
bin edges used: {'income': [64747.42194529035, 69585.1661527817, 74280.16364993049], 'age': [37.82212251253142, 42.362698203748764, 45.83481033895491]}
tolerance cutoff: 0.5656685620472034


## 5. Try it on a CSV

`fit_match` accepts a file path. CSV/Parquet/Feather stream through DuckDB; Excel is loaded then streamed.

In [9]:
csv_path = root / "demo_input.csv"
df.to_csv(csv_path, index=False)
from_file = ControlMatcher(config).fit_match(str(csv_path))
from_file.coverage_summary

{'n_target': 200,
 'n_matched': 149,
 'n_no_control': 0,
 'n_below_tolerance': 37,
 'n_unmatched': 14,
 'n_thin_stratum': 13,
 'pct_matched': 0.745,
 'pct_no_control': 0.0,
 'pct_below_tolerance': 0.185,
 'tolerance_cutoff': 0.5656685620472034}

## 6. Knobs worth turning

| knob | effect |
|---|---|
| `n=2` | up to two controls per target (still no reuse) |
| `tolerance=0.8` | keep only the strongest 20% of pairs |
| `n_bins=6` | finer strata, more `no_control_available` / `thin_stratum` |
| `weights={"income": 3}` | income dominates distance |
| `min_control_pool_size=10` | more thin flags, matching still runs |
| `n_workers=4` | score strata on 4 threads (identical results, opt-in) |
| `duckdb_threads=2` | hand execution threads to DuckDB (opt-in) |
| `progress=True` | tqdm bars on stderr when a TTY (needs the `progress` extra) |

Change a value below and re-run the cell.

In [10]:
strict = MatchConfig(
    match_vars=["income", "age", "region"],
    treatment_col="is_target",
    id_col="id",
    n=1,
    tolerance=0.8,
    min_control_pool_size=5,
    n_bins=4,
    n_workers=2,          # opt-in: parallel per-stratum scoring
    duckdb_threads=2,     # opt-in: DuckDB execution threads
    progress=True,        # opt-in: bars on stderr when a TTY
)
strict_result = ControlMatcher(strict).fit_match(df)
strict_result.coverage_summary

pipeline:   0%|          | 0/10 [00:00<?, ?it/s]

score:   0%|          | 0/62 [00:00<?, ?it/s]

match:   0%|          | 0/2105 [00:00<?, ?it/s]

{'n_target': 200,
 'n_matched': 38,
 'n_no_control': 0,
 'n_below_tolerance': 148,
 'n_unmatched': 14,
 'n_thin_stratum': 46,
 'pct_matched': 0.19,
 'pct_no_control': 0.0,
 'pct_below_tolerance': 0.74,
 'tolerance_cutoff': 0.9014048910196473}